In [ ]:
# imports - ignore
# !pip install gymnasium stable_baselines3[extra] opencv-python highway_env

# Documentation for Trajectory Distribution Fuzzing in Highway Roundabout

**Note:** This framework is designed to run **rollouts** (episodes) under configurable distributions. You can define your own scenario parameters by creating a `ScenarioParams` object with your desired distributions and passing it to the environment via `gym.make(..., scenario_params=my_params)`. If you want the nominal (default) distribution, simply omit the argument or pass `None`—the environment will use a built‑in nominal set.

For the format of the `ScenarioParams` object, please check out the `setup` variable later in the code. You may define each item as a tensor or a nn.Parameter object.

This document provides an overview and explanation of the code implemented in `traj_distribution.ipynb`. The notebook extends the `highway-env` roundabout environment to support **learnable/fuzzable scenario parameters** that control various disturbances in observation, action, and environment dynamics. It also includes a custom vehicle with a **politeness parameter** drawn from a Beta distribution, and a `rollout` function that computes the log-probability of a trajectory under the defined parameter distributions.

## Table of Contents
1. [Parameter Classes](#parameter-classes)
2. [Helper Functions](#helper-functions)
3. [Custom Vehicle: `CustomPoliteVehicle`](#custom-vehicle-custompolitevehicle)
4. [Simulated Environment: `SimulatedEnv`](#simulated-environment-simulatedenv)
5. [Rollout Function](#rollout-function)
6. [Usage Example](#usage-example)
7. [Dependencies](#dependencies)

---

## Parameter Classes

These dataclasses encapsulate the learnable/fuzzable parameters. They are designed to hold either fixed values or PyTorch tensors/parameters that can be optimized.

```python
from typing import Union, List
import torch
from dataclasses import dataclass

ParamType = Union[float, int, List[float], torch.Tensor, torch.nn.Parameter]
```

### `GaussianMixtureParam`
Represents a mixture of Gaussians:
- `p`: list of probabilities for the first `n-1` components (the last is `1 - sum(p)`).
- `mu`: list of means for each component.
- `sigma`: list of standard deviations for each component.

### `NormalParam`
A single Gaussian:
- `mu`: mean.
- `sigma`: standard deviation.

### `ProbabilityParam`
A categorical probability (or set of probabilities):
- `p`: list of probabilities (length `n-1` for `n` categories, or length 1 for a binary).

### `BetaParam`
Parameters for a Beta distribution:
- `ab`: two-element list/tensor `[alpha, beta-alpha]` for Beta Distribution.

### `ScenarioParams`
Collects all scenario parameters:
- **Observation disturbances**:
  - `initial_position_x`, `initial_position_y`: GaussianMixtureParam for position noise.
  - `velocity_x`, `velocity_y`: GaussianMixtureParam for velocity noise.
- **Action disturbances**:
  - `high_lvl_ctrl_noise`: ProbabilityParam for high‑level control noise.
  - `initial_speed`: NormalParam for ego vehicle’s initial speed (clipped to `[0,16]`).
  - `initial_heading`: ProbabilityParam for ego heading. This is a length-3 array of the probabilities for east, north, and west (we derive south by 1 - sum of all other probabilities)
- **Environment disturbances**:
  - `politeness`: BetaParam for the politeness of other vehicles.
  - `other_vehicle_speed`: NormalParam for the speed of other vehicles (clipped to `[0,32]`).
  - `entering_vehicle_position`: GaussianMixtureParam for the longitudinal position of the entering vehicle.

---

## Helper Functions

### `to_tensor(param: ParamType, dtype=torch.float32) -> torch.Tensor`
Converts a scalar, list, `nn.Parameter`, or tensor to a torch tensor of the specified dtype.

### `sample_gaussian_mixture(param: GaussianMixtureParam) -> torch.Tensor`
Samples a single value from the Gaussian mixture:
- Normalizes probabilities.
- Selects a component via categorical sampling.
- Returns a sample from the chosen Gaussian.

### `gaussian_pdf(x, mu, sigma)`
Computes the probability density function of a univariate Gaussian at `x`.

### `gaussian_mixture_pdf(x, param: GaussianMixtureParam)`
Computes the PDF of a Gaussian mixture at `x` using the stored `p`, `mu`, `sigma`. Automatically pads `p` if necessary.

---

## Custom Vehicle: `CustomPoliteVehicle`

Inherits from `IDMVehicle` (Intelligent Driver Model) and adds a **politeness parameter** drawn from a Beta distribution.

**Key points**:
- Constructor accepts `politeness` (interpreted as `[alpha, beta-alpha]` in the code – note the unconventional mapping).  
  - `alpha = politeness[0]`
  - `beta = alpha + politeness[1]`
- A differentiable sample is drawn using `Beta(alpha, beta).rsample()` and stored as `self.POLITENESS`.
- Overrides `make_on_lane` class method to accept `politeness` and pass it to the constructor. This preserves the original `make_on_lane` functionality while adding the extra argument.

```python
@classmethod
def make_on_lane(cls, road, lane_index, longitudinal, speed=None, politeness=None):
    lane = road.network.get_lane(lane_index)
    if speed is None:
        speed = lane.speed_limit
    return cls(road, lane.position(longitudinal, 0), lane.heading_at(longitudinal), speed, politeness)
```

---

## Simulated Environment: `SimulatedEnv`

Inherits from `RoundaboutEnv` and modifies vehicle creation to incorporate the scenario parameters.

### Constructor
- Accepts optional `config` and `scenario_params`.
- If `scenario_params` is `None`, a default nominal set is created.
- Stores `self.scenario_params` **before** calling `super().__init__()` to ensure it is available during `_make_vehicles` (which is called by the base class `reset`).

### `_make_vehicles`
- **Ego vehicle**:
  - Initial speed sampled from `initial_speed` (clipped to 0–16).
  - Heading sampled from `initial_heading` (four destinations: `"exr"`, `"nxr"`, `"wxr"`, `"sxr"`).
  - The sampled heading index is stored as `ego_vehicle.heading_idx` for later log‑prob calculation.
- **Other vehicles** (incoming, additional, entering):
  - Created using `CustomPoliteVehicle.make_on_lane`.
  - Speed sampled from `other_vehicle_speed` (clipped to 0–32).
  - Politeness passed as `self.scenario_params.politeness.ab`.
  - Destinations randomly chosen from `["exr", "sxr", "nxr"]`.
- **Entering vehicle**:
  - Longitudinal position sampled from `entering_vehicle_position` (Gaussian mixture).
- All vehicles are appended to `self.road.vehicles`.

The environment is registered with Gymnasium as `"SimulatedEnv-v0"`.

---

## Rollout Function

The `rollout(env)` function runs one episode using a trained policy (e.g., DQN) while **adding noise** to observations and actions according to the scenario parameters. It also computes the **log‑probability** of the observed trajectory under the parameter distributions.

**Steps**:

1. **Sample observation noise** for each vehicle (position and velocity) from the corresponding GaussianMixtureParams.
2. **Compute log‑prob contributions** for:
   - Speeds of other vehicles (Normal distribution).
   - Politeness values of other vehicles (Beta distribution).
   - Entering vehicle’s longitudinal position (Gaussian mixture).
   - Ego vehicle’s initial speed (Normal) and heading (Categorical).
3. **Inside the episode loop**:
   - Add the pre‑sampled noise to the observations.
   - Query the model for an action (deterministic).
   - With probability `high_lvl_ctrl_noise`, replace the action with a random available action.
   - Add log‑prob for the action choice (`log(p_noise/|A|)` if random, else `log(1-p_noise)`).
   - Add log‑prob for the noisy observations (using the corresponding PDFs).
   - Record the noisy observation.
4. Return a dictionary containing the total `log_prob`.

The function assumes the environment is wrapped with a `RecordVideo` wrapper for rendering.

---

## Usage Example

```python
# Define scenario parameters (nominal)
setup = ScenarioParams(...)   # see notebook

# Create environment with custom parameters
env = gym.make("SimulatedEnv-v0", render_mode="rgb_array", scenario_params=setup)

# Train a DQN agent (example)
model = DQN("MlpPolicy", env, ...)
model.learn(total_timesteps=10000)

# Record videos and compute log-probabilities
env = RecordVideo(env, video_folder="videos/")
logs = []
for _ in range(5):
    logs.append(rollout(env))
```

---

## Dependencies

- Python 3.12+
- `gymnasium`
- `stable_baselines3`
- `highway-env`
- `torch`
- `numpy`
- `opencv-python` (for video recording)

---

## Notes

- The politeness mapping in `CustomPoliteVehicle` uses a non‑standard parameterization (`beta = alpha + politeness[1]`). This may be a custom choice for differentiability; adjust if needed.
- The `rollout` function currently assumes a fixed number of vehicles (4 other vehicles). For generalization, you might want to query the actual number from `env.unwrapped.road.vehicles`.
- The code is designed for **fuzzing** and **robustness evaluation** by perturbing parameters and computing trajectory probabilities. It can be integrated into optimisation loops (e.g., Bayesian optimisation) to find worst‑case scenarios.

For further details, refer to the inline comments in the notebook.

In [1]:
import gymnasium as gym
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import DQN

import highway_env  # noqa: F401

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
objc[75816]: Class SDLApplication is implemented in both /Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x2c6688890) and /Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x30fabd2c8). One of the two will be used. Which one is undefined.
objc[75816]: Class SDLAppDelegate is implemented in both /Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x2c66888e0) and /Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x30fabd318). One of the two will be used. W

In [ ]:
# variables to tweak

# Observation
# initial position: Initial Position of Vehicles
# Gaussian Mixture: p N(mu_1, sigma_1^2) + (1-p) N(mu_2, sigma_2^2) --> Learn: p_obs_pos, mu_obs_pos, sigma_obs_pos
# Observed Velocity of Vehicles: p N(mu_1, sigma_1^2) + (1-p) N(mu_2, sigma_2^2) --> Learn: p_obs_vel, mu_obs_vel, mu_obs_pos

# Action
# high level speed control: p for controlling randomness
# initial speed: N(mu, sigma) # use truncated normal
# initial heading: p_e, p_n, p_w for heading east, north, west (1-p_e-p_n-p_w for south)

# Environment
# politeness: Beta(a,a+b)
# other cars speed: Normal(mu, sigma) # use truncated normal
# initial location of entering car: p N(mu_1, sigma_1^2) + (1-p) N(mu_2, sigma_2^2)


In [ ]:
from typing import Union, List, Dict
import torch
import math
from dataclasses import dataclass, field

# Allowed types for learnable parameters
ParamType = Union[float, int, List[float], torch.Tensor, torch.nn.Parameter]

@dataclass
class GaussianMixtureParam:# mixture between n gaussians
    p: ParamType # length n-1
    mu: ParamType # length n list or tensor
    sigma: ParamType # length n list or tensor

@dataclass
class NormalParam:
    mu: ParamType # length 1
    sigma: ParamType # length 1

@dataclass
class ProbabilityParam:
    p: ParamType # length 1 or n-1

@dataclass
class BetaParam:
    ab: ParamType

@dataclass
class ScenarioParams:
    # Observation
    initial_position_x: GaussianMixtureParam
    initial_position_y: GaussianMixtureParam
    velocity_x: GaussianMixtureParam
    velocity_y: GaussianMixtureParam

    # Action
    high_lvl_ctrl_noise: ProbabilityParam  
    initial_speed: NormalParam # truncate between 0 and 16
    initial_heading: ProbabilityParam

    # Environment
    politeness: BetaParam
    other_vehicle_speed: NormalParam # truncate between 0 and somewhere like 32
    entering_vehicle_position: GaussianMixtureParam
    
SQRT_2 = math.sqrt(2)

# Here is an example of how to use
nominal = ScenarioParams(
    # observation disturbances
    initial_position_x=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    initial_position_y=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    velocity_x=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
    ),
    velocity_y=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
    ),
    # action disturbance
    high_lvl_ctrl_noise=ProbabilityParam(p = [0.0]),
    initial_speed=NormalParam(mu=8.0, sigma=0.5),
    initial_heading=ProbabilityParam(p = [0.0, 1.0, 0.0]),
    
    # environment disturbance
    politeness=BetaParam(ab= [1,1]), # uniform distribution
    other_vehicle_speed=NormalParam(mu=16.0, sigma=2.0),
    entering_vehicle_position=GaussianMixtureParam(
        p=[1.0],
        mu=[5.0, 5.0], # tip for fuzzing: mu = 100.0 can cause more crashes
        sigma=[2.0, 2.0]
    )
)

In [124]:
# helper functions
import torch.nn as nn
def to_tensor(param: ParamType, dtype=torch.float32) -> torch.Tensor:
    """Convert scalar, list, or nn.Parameter to a torch tensor."""
    if isinstance(param, torch.Tensor):
        return param.to(dtype)
    if isinstance(param, nn.Parameter):
        return param.data.to(dtype)
    if isinstance(param, (list, tuple)):
        return torch.tensor(param, dtype=dtype)
    return torch.tensor([param], dtype=dtype)  # scalar -> 1-element tensor

def sample_gaussian_mixture(param: GaussianMixtureParam) -> torch.Tensor:
    p = to_tensor(param.p)
    mu = to_tensor(param.mu)
    sigma = to_tensor(param.sigma)

    # Pad mixture probabilities if necessary
    if len(p) < len(mu):
        last_prob = 1.0 - p.sum()
        p = torch.cat([p, last_prob.unsqueeze(0)])

    # Choose a mixture component
    mixture_idx = torch.distributions.Categorical(p).sample().item()

    # Sample from selected Gaussian
    sample = mu[mixture_idx] + sigma[mixture_idx] * torch.randn(1)
    return sample.squeeze()

def gaussian_pdf(x, mu, sigma):
    # scalar
    return (1.0 / (sigma * math.sqrt(2*math.pi))) * torch.exp(-0.5 * ((x - mu)/sigma)**2)

def gaussian_mixture_pdf(x, param: GaussianMixtureParam):
    p = torch.tensor(param.p)
    mu = torch.tensor(param.mu)
    sigma = torch.tensor(param.sigma)
    
    # Pad probabilities if needed
    if len(p) < len(mu):
        last_prob = 1.0 - p.sum()
        p = torch.cat([p, last_prob.unsqueeze(0)])
        
    pdf = 0.0
    for w, m, s in zip(p, mu, sigma):
        pdf += w * gaussian_pdf(x, m, s)
    # print(pdf)
    return pdf

In [142]:
# for new roundabout
from highway_env.envs.roundabout_env import RoundaboutEnv as OriginalRoundabout
from highway_env import utils
from highway_env.envs.common.abstract import AbstractEnv
from highway_env.road.lane import CircularLane, LineType, SineLane, StraightLane
from highway_env.road.road import Road, RoadNetwork
from highway_env.vehicle.controller import MDPVehicle
import numpy as np 
from typing import Optional,  Tuple
from highway_env.vehicle.behavior import IDMVehicle
import torch.nn as nn
from gymnasium.envs.registration import register
from highway_env.vehicle.objects import RoadObject
LaneIndex = Tuple[str, str, int]
class CustomPoliteVehicle(IDMVehicle):
    # def __init__(self, *args, **kwargs, politeness=None):
    #     super().__init__(*args, **kwargs)
    #     # Default nominal values
    #     if politeness is None:
    #         politeness = torch.tensor([1.0, 1.0], dtype=torch.float)

    #     # Ensure it's a tensor
    #     if isinstance(politeness, nn.Parameter):
    #         tensor_p = politeness
    #     else:
    #         tensor_p = torch.tensor(politeness, dtype=torch.float)

    #     # Separate alpha and beta: alpha = tensor_p[0], beta = tensor_p[1]
    #     alpha = tensor_p[0]
    #     beta = alpha + tensor_p[1]  # second element is beta-alpha

    #     # Differentiable sample
    #     dist = torch.distributions.Beta(alpha, beta)
    #     self.POLITENESS = dist.rsample()  # keep as tensor for gradient
    def __init__(self, road, position, heading, speed, politeness=None, **kwargs):
        super().__init__(road, position, heading, speed, **kwargs)
        if politeness is None:
            politeness = torch.tensor([1.0, 1.0], dtype=torch.float)
        # ensure tensor
        if isinstance(politeness, nn.Parameter):
            tensor_p = politeness
        else:
            tensor_p = torch.tensor(politeness, dtype=torch.float)
        alpha = tensor_p[0]
        beta = alpha + tensor_p[1]
        dist = torch.distributions.Beta(alpha, beta)
        self.POLITENESS = dist.rsample()
    @classmethod
    def make_on_lane(
        cls,
        road: Road,
        lane_index: LaneIndex,
        longitudinal: float,
        speed: float | None = None,
        politeness: BetaParam | None = None,
    ) -> RoadObject:
        """
        Create a vehicle on a given lane at a longitudinal position.

        :param road: a road object containing the road network
        :param lane_index: index of the lane where the object is located
        :param longitudinal: longitudinal position along the lane
        :param speed: initial speed in [m/s]
        :return: a RoadObject at the specified position
        """
        lane = road.network.get_lane(lane_index)
        if speed is None:
            speed = lane.speed_limit
        return cls(
            road, lane.position(longitudinal, 0), lane.heading_at(longitudinal), speed, politeness
        )

class SimulatedEnv(OriginalRoundabout):
    # @classmethod
    def __init__(self, config=None, scenario_params: Optional[ScenarioParams] = None, render_mode: str | None = None):
        """
        Constructor for the simulated roundabout environment.

        Args:
            config: Optional dictionary of environment configuration parameters (passed to OriginalRoundabout).
            scenario_params: Optional ScenarioParams object defining learnable/fuzzable parameters.
        """
        
        if scenario_params is None:
            scenario_params = ScenarioParams(
                # observation disturbances
                initial_position_x=GaussianMixtureParam(
                    p=[1.0], 
                    mu=[0.0, 0.0], 
                    sigma=[0.005, 0.005]
                ),
                initial_position_y=GaussianMixtureParam(
                    p=[1.0], 
                    mu=[0.0, 0.0], 
                    sigma=[0.005, 0.005]
                ),
                velocity_x=GaussianMixtureParam(
                    p=[1.0],
                    mu=[0.0, 0.0],
                    sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
                ),
                velocity_y=GaussianMixtureParam(
                    p=[1.0],
                    mu=[0.0, 0.0],
                    sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
                ),
                # action disturbance
                high_lvl_ctrl_noise=ProbabilityParam(p = [0.0]),
                initial_speed=NormalParam(mu=8.0, sigma=0.5),
                initial_heading=ProbabilityParam(p = [0.0, 1.0, 0.0]),
                
                # environment disturbance
                politeness=BetaParam(ab= [1,1]), # uniform distribution
                other_vehicle_speed=NormalParam(mu=16.0, sigma=2.0),
                entering_vehicle_position=GaussianMixtureParam(
                    p=[1.0],
                    mu=[5.0, 5.0], # tip for fuzzing: mu = 100.0 can cause more crashes
                    sigma=[2.0, 2.0]
                )
            )
        self.scenario_params = scenario_params
        super().__init__(config=config, render_mode=render_mode)
        
    def _make_vehicles(self) -> None:
        position_deviation = 2.0
        min_speed, max_speed = 0.0, 32.0
        # speed_deviation = 2.0

        # Ego-vehicle
        initial_mu = torch.as_tensor(self.scenario_params.initial_speed.mu)
        initial_sigma = torch.as_tensor(self.scenario_params.initial_speed.sigma)
        initial_speed = torch.clamp(initial_mu + 
                    initial_sigma * 
                    torch.randn_like(initial_mu), 
                    min=min_speed, max=16.0)
        ego_lane = self.road.network.get_lane(("ser", "ses", 0))
        ego_vehicle = self.action_type.vehicle_class(
            self.road,
            ego_lane.position(125.0, 0.0),
            # speed=8.0,
            speed = float(initial_speed),
            heading=ego_lane.heading_at(140.0),
        )
        try:
            # ego_vehicle.plan_route_to("nxr") # default: "nxr"
            
            # ego_destinations = ["exr", "sxr", "nxr", "wxr"]
            # destination = self.np_random.choice(ego_destinations)
            # ego_vehicle.plan_route_to(destination)
            p_tensor = torch.tensor(self.scenario_params.initial_heading.p, dtype=torch.float32)
            p_s = 1.0 - torch.sum(p_tensor)  # last probability
            probs = torch.cat([p_tensor, p_s.unsqueeze(0)])  # shape [4]

            # sample using categorical distribution
            cat = torch.distributions.Categorical(probs)
            heading_idx = cat.sample()  # tensor scalar

            destination = ["exr", "nxr", "wxr", "sxr"][heading_idx.item()]
            ego_vehicle.heading_idx = heading_idx # store this
            ego_vehicle.plan_route_to(destination)
        except AttributeError:
            pass
        self.road.vehicles.append(ego_vehicle)
        self.vehicle = ego_vehicle

        # Incoming vehicle
        destinations = ["exr", "sxr", "nxr"]
        other_vehicles_type = CustomPoliteVehicle #utils.class_from_path(self.config["other_vehicles_type"])
        other_mu = torch.as_tensor(self.scenario_params.other_vehicle_speed.mu)
        other_sigma = torch.as_tensor(self.scenario_params.other_vehicle_speed.sigma)
        other_car_speed =  torch.clamp(other_mu + 
                            other_sigma * 
                            torch.randn_like(other_mu), 
                            min=min_speed, max=max_speed)
        vehicle = other_vehicles_type.make_on_lane(
            road = self.road,
            lane_index = ("we", "sx", 1),
            longitudinal=50.0 + self.np_random.normal() * position_deviation,
            # speed=16 + self.np_random.normal() * speed_deviation,
            # speed: othervehicle speed
            
            speed = float(other_car_speed),
            politeness = self.scenario_params.politeness.ab
        )
        

        if self.config["incoming_vehicle_destination"] is not None:
            destination = destinations[self.config["incoming_vehicle_destination"]]
        else:
            destination = self.np_random.choice(destinations)
        vehicle.plan_route_to(destination)
        vehicle.randomize_behavior()
        self.road.vehicles.append(vehicle)

        # Other vehicles 
        for i in list(range(1, 2)) + list(range(-1, 0)):
            vehicle = other_vehicles_type.make_on_lane(
                road = self.road,
                lane_index = ("we", "sx", 0),
                longitudinal=20.0 * float(i)
                + self.np_random.normal() * position_deviation,
                # speed=16.0 + self.np_random.normal() * speed_deviation,
                speed = float(other_car_speed),
                politeness = self.scenario_params.politeness.ab
                
            )
            vehicle.plan_route_to(self.np_random.choice(destinations))
            vehicle.randomize_behavior()
            self.road.vehicles.append(vehicle)

        # Entering vehicle
        # Calculate entering vehicle's position
        entering_position = 0
        last_prob = 1
        num_mixtures_entering = len(self.scenario_params.entering_vehicle_position.mu)
        entering_pos_mu = torch.as_tensor(self.scenario_params.entering_vehicle_position.mu)
        entering_pos_sigma = torch.as_tensor(self.scenario_params.entering_vehicle_position.sigma)
        for idx in range(num_mixtures_entering - 1):
            entering_position += self.scenario_params.entering_vehicle_position.p[idx]*(
                entering_pos_mu[idx] + 
                entering_pos_sigma[idx] * 
                torch.randn_like(entering_pos_mu[idx])
            )
            last_prob -= self.scenario_params.entering_vehicle_position.p[idx]
        entering_position += last_prob*(
            entering_pos_mu[num_mixtures_entering-1] + 
            entering_pos_sigma[num_mixtures_entering-1] * 
            torch.randn_like(entering_pos_mu[num_mixtures_entering-1])
        )
        
        vehicle = other_vehicles_type.make_on_lane(
            road = self.road,
            lane_index = ("eer", "ees", 0),
            longitudinal= entering_position.item(), #5.0 + self.np_random.normal() * position_deviation,
            # speed=16.0 + self.np_random.normal() * speed_deviation,
            speed = float(other_car_speed),
            politeness = self.scenario_params.politeness.ab
                
        )
        vehicle.plan_route_to(self.np_random.choice(destinations))
        vehicle.randomize_behavior()
        self.road.vehicles.append(vehicle)
    # def _rollout(self):
        

In [143]:

register(
    id="SimulatedEnv-v0",
    entry_point="__main__:SimulatedEnv",  # or the module path if in another file
)


/Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment SimulatedEnv-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [ ]:
# Here is an example of how to use.
setup = ScenarioParams(
    # observation disturbances
    initial_position_x=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    initial_position_y=GaussianMixtureParam(
        p=[1.0], 
        mu=[0.0, 0.0], 
        sigma=[0.005, 0.005]
    ),
    velocity_x=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
    ),
    velocity_y=GaussianMixtureParam(
        p=[1.0],
        mu=[0.0, 0.0],
        sigma=[SQRT_2 * 0.005/0.1, SQRT_2 * 0.005/0.1]
    ),
    # action disturbance
    high_lvl_ctrl_noise=ProbabilityParam(p = [0.0]),
    initial_speed=NormalParam(mu=8.0, sigma=0.5), # keep <= 16
    initial_heading=ProbabilityParam(p = [0.0, 1.0, 0.0]),
    
    # environment disturbance
    politeness=BetaParam(ab= [1,1]), # uniform distribution
    other_vehicle_speed=NormalParam(mu=16.0, sigma=2.0),
    entering_vehicle_position=GaussianMixtureParam(
        p=[1.0],
        mu=[5.0, 5.0], # tip for fuzzing: mu = 100.0 can cause more crashes
        sigma=[2.0, 2.0]
    )
)

In [145]:
TRAIN = False

# Create the environment
# env = gym.make("roundabout-v0", render_mode="rgb_array")#, roundabout_radius = 30)
env = gym.make("SimulatedEnv-v0", render_mode="rgb_array", scenario_params = setup)#, roundabout_radius = 30)

# add disturbances



obs, info = env.reset()

# Create the model
model = DQN(
    "MlpPolicy",
    env,
    policy_kwargs=dict(net_arch=[256, 256]),
    learning_rate=5e-4,
    buffer_size=15000,
    learning_starts=200,
    batch_size=32,
    gamma=0.8,
    train_freq=1,
    gradient_steps=1,
    target_update_interval=50,
    verbose=1,
    tensorboard_log="roundabout_dqn/",
)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [160]:
from src.robustness import compute_robustness, weights_from_vector
def rollout(env,robustness_weights = [0.2042, 0.1982, 0.4393, 0.0111, 0.1471], eps = 1e-10,  seed = None, speed_scale = 16.0):
    log_prob = 0.0
    # Assuming inside a class or you have access to env.unwrapped.scenario_params
    p_noise = env.unwrapped.scenario_params.high_lvl_ctrl_noise.p
    # print(p_noise)
    if isinstance(p_noise, torch.nn.Parameter) or torch.is_tensor(p_noise):
        p_noise = torch.clamp(p_noise, 0.0, 1.0).item()  # ensure it's a float between 0 and 1
    else:
        p_noise = p_noise[0]


    # Car noise:
    num_cars = 4
    noise_position_x = torch.tensor([sample_gaussian_mixture(env.unwrapped.scenario_params.initial_position_x) for _ in range(num_cars)])
    noise_position_y = torch.tensor([sample_gaussian_mixture(env.unwrapped.scenario_params.initial_position_y) for _ in range(num_cars)])
    noise_velocity_x = torch.tensor([sample_gaussian_mixture(env.unwrapped.scenario_params.velocity_x) for _ in range(num_cars)])
    noise_velocity_y = torch.tensor([sample_gaussian_mixture(env.unwrapped.scenario_params.velocity_y) for _ in range(num_cars)])
    # probs = to_tensor(env.unwrapped.scenario_params.initial_heading.p)
    # probs

    # 1. Environment parameters (sampled once)
    env_params = env.unwrapped.scenario_params
    for i in range(1, 5):
        v = torch.tensor(env.unwrapped.road.vehicles[i].speed)  # example: other_vehicle_speed
        # print(env.unwrapped.road.vehicles)
        log_prob += torch.log(torch.clamp(torch.distributions.Normal(env_params.other_vehicle_speed.mu,
                                                        env_params.other_vehicle_speed.sigma).log_prob(v) + eps, eps))

        # Add politeness probability
        pol_tensor = torch.tensor(env_params.politeness.ab, dtype=torch.float)
        alpha, beta = pol_tensor[0], pol_tensor[0] + pol_tensor[1]
        log_prob += torch.distributions.Beta(alpha, beta).log_prob(env.unwrapped.road.vehicles[i].POLITENESS)
    # Entering car
    log_prob += torch.log(torch.clamp(
        gaussian_mixture_pdf(env.unwrapped.road.vehicles[-1].position[0],
                            env_params.entering_vehicle_position) + eps, eps)
    )
    
    # Consider ego car's initial speed and direction
    # Ego speed
    log_prob += torch.distributions.Normal(
        env_params.initial_speed.mu, env_params.initial_speed.sigma
    ).log_prob(torch.tensor(env.unwrapped.vehicle.speed))

    # Ego heading
    # You need to save the sampled heading index during _make_vehicles
    head_probs = torch.tensor(env_params.initial_heading.p, dtype=torch.float32)
    last_prob = 1.0 - head_probs.sum()
    probs = torch.cat([head_probs, last_prob.unsqueeze(0)])  # shape [4]
    log_prob += torch.log(probs[env.unwrapped.road.vehicles[0].heading_idx] + eps)
    
    trajectory = []
    done = truncated = False
    obs, info = env.reset()
    # print(log_prob)
    
    # logging robustness
    velocities = []
    min_dist = float("inf")
    lane_changes = 0
    on_road_steps = 0
    total_steps = 0
    cumulative_reward = 0.0
    prev_action = None

    unwrapped = env.unwrapped
    ego = unwrapped.vehicle
    policy_freq = unwrapped.config.get("policy_frequency", 1)
    while not (done or truncated):
        # Predict
        # add sensor noise:
        obs_noisy = obs.copy()
        for i in range(num_cars):
            obs_noisy[i+1, 1] += noise_position_x[i]
            obs_noisy[i+1, 2] += noise_position_y[i]
            obs_noisy[i+1, 3] += noise_velocity_x[i]
            obs_noisy[i+1, 4] += noise_velocity_y[i]

        action, _states = model.predict(obs_noisy, deterministic=True)
        
        # # # action fuzz
        
        # if random.random() < 0.4:
        #     available = env.unwrapped.action_type.get_available_actions()
        #     print(available)
        #     action = random.choice(available)
        # High-level action fuzzing
        if torch.rand(1).item() < p_noise:
            available = env.unwrapped.action_type.get_available_actions()
            action = env.unwrapped.np_random.choice(available)
            action_prob = p_noise / len(available)
        else:
            action_prob = 1.0 - p_noise
        log_prob += torch.log(torch.clamp(torch.tensor(action_prob) + eps, eps))
        for i in range(1, num_cars+1):
            log_prob += torch.log(torch.clamp(gaussian_mixture_pdf(obs_noisy[i,1], env_params.initial_position_x) + eps, eps))
            log_prob += torch.log(torch.clamp(gaussian_mixture_pdf(obs_noisy[i,2], env_params.initial_position_y) + eps, eps))
            log_prob += torch.log(torch.clamp(gaussian_mixture_pdf(obs_noisy[i,3], env_params.velocity_x) + eps, eps))
            log_prob += torch.log(torch.clamp(gaussian_mixture_pdf(obs_noisy[i,4], env_params.velocity_y) + eps, eps))

        trajectory.append(obs_noisy)

        # # Get reward
        obs, reward, done, truncated, info = env.step(action)
        
        cumulative_reward += float(reward)
        total_steps += 1
        ego = unwrapped.vehicle
        v = np.sqrt(ego.velocity[0] ** 2 + ego.velocity[1] ** 2)
        velocities.append(v)

        for vehicle in unwrapped.road.vehicles:
            if vehicle is not ego:
                d = np.linalg.norm(ego.position - vehicle.position)
                min_dist = min(min_dist, d)

        if ego.on_road:
            on_road_steps += 1

        a = int(np.asarray(action).flat[0])
        if prev_action is not None and a != prev_action and a in [0, 2]:
            lane_changes += 1
        prev_action = a
        
        # Render
        env.render()
    
    
    velocities = np.array(velocities)
    dt = 1.0 / policy_freq
    accel = np.diff(velocities) / dt
    jerk = np.diff(accel) / dt

    decel = np.clip(-accel, 0, None)
    n = len(decel)
    brake_severity = (
        float(np.sum(decel ** 2) / n) if n > 0 else 0.0
    )

    mean_vel = np.mean(velocities) if len(velocities) > 0 else 0.0
    jerk_score = float(np.mean(jerk ** 2)) if len(jerk) > 0 else 0.0
    on_road_frac = on_road_steps / max(total_steps, 1)
    trajectory_metrics = {
        "success": 0.0 if unwrapped.vehicle.crashed else 1.0,
        "min_distance": min_dist if min_dist != float("inf") else 0.0,
        "avg_speed_ratio": mean_vel / speed_scale,
        "jerk_score": jerk_score,
        "brake_severity": brake_severity,
        "lane_changes": lane_changes,
        "on_road_frac": on_road_frac,
        "cumulative_reward": cumulative_reward,
    }
    env.close()
    return {'log_prob': log_prob.item(), 'robustness': compute_robustness(trajectory_metrics, weights = robustness_weights)}

In [161]:
import random
# Run the trained model and record video
model = DQN.load("roundabout_dqn/model", env=env, device="cpu")
# model.set_env(env)
env = RecordVideo(
    env, video_folder="roundabout_dqn/videos/", episode_trigger=lambda e: True
)
env.unwrapped.config["simulation_frequency"] = 15  # Higher FPS for rendering
env.unwrapped.config.update({
     "observation": {
         # default
                    "type": "Kinematics",
                    "absolute": True,
                    
                    "features_range": {
                        "x": [-100, 100],
                        "y": [-100, 100],
                        "vx": [-15, 15], #[-15, 15],
                        "vy": [-15, 15], #[-15, 15],
                    },
                    
                    # # add new
                    "vehicles_count": 5,
                    # "order": "shuffled",
                    
                },
     
     
    # "sensor_noise": 0.02
})
# low_level_agent_fuzzing(env.unwrapped.vehicle, acc_std=1, steer_std=5)

env.unwrapped.set_record_video_wrapper(env)

logs = []
for videos in range(2):
    logs.append(rollout(env, robustness_weights = weights_from_vector([0.2042, 0.1982, 0.4393, 0.0111, 0.1471]), eps = 1e-10)) # store log_probs and robustness

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/dkoffical/envs/myenv312/lib/python3.12/site-packages/gymnasium/wrappers/rendering.py:293: UserWarning: WARN: Overwriting existing videos at /Users/dkoffical/Documents/GitHub/StanfordRoundabout/roundabout_dqn/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


In [162]:
print(logs)

[{'log_prob': -1145.7880859375, 'robustness': 0.7983793380911535}, {'log_prob': -1566.270751953125, 'robustness': 0.8743406556051474}]


In [ ]:
# !pip freeze > colab_requirements.txt

In [ ]:
# imports
# # import numpy
# # import gymnasium
# # import stable_baselines3
# # import cv2
# # import highway_env
# import torch
# import torchvision
# import torchaudio


# # print("numpy:", numpy.__version__)
# # print("gymnasium:", gymnasium.__version__)
# # print("stable_baselines3:", stable_baselines3.__version__)
# # print("opencv-python:", cv2.__version__)
# # print("highway_env:", highway_env.__version__)

# print("torch:", torch.__version__)
# print("torchvision:", torchvision.__version__)
# print("torchaudio:", torchaudio.__version__)


numpy: 2.0.2
gymnasium: 1.2.3
stable_baselines3: 2.7.1
opencv-python: 4.13.0
highway_env: 1.10.2
